<a href="https://colab.research.google.com/github/ArthurrCr/cloudband/blob/main/notebooks/00_baselines/score_ocm_cloudsen12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --quiet "tacoreader<1.0" omnicloudmask==1.7.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 112.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 12.4 MB/s eta 0:00:00


In [5]:
REPO_URL = "https://github.com/ArthurrCr/cloudband.git"
PROJECT_DIR = "/content/cloudband"
BRANCH = "main"

import os
import sys

if not os.path.exists(PROJECT_DIR):
    !git clone --quiet {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git fetch --quiet origin && git reset --quiet --hard origin/{BRANCH}

SRC_DIR = f"{PROJECT_DIR}/src"
os.chdir(PROJECT_DIR)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

!PYTHONPATH={SRC_DIR} python -m pytest tests -q

........................................................................ [ 77%]
.....................                                                    [100%]
=============================== warnings summary ===============================
tests/contract/test_pipeline.py::test_wrong_raster_size_is_rejected
tests/contract/test_pipeline.py::test_duplicate_predictions_are_rejected
tests/contract/test_pipeline.py::test_attach_and_score_round_trip
tests/contract/test_pipeline.py::test_pooled_counts_equal_whole_collection
  /usr/local/lib/python3.13/dist-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
    dataset = writer(

tests/contract/test_pipeline.py::test_wrong_raster_size_is_rejected
tests/contract/test_pipeline.py::test_attach_and_score_round_trip
tests/contract/test_pipeline.py::test_pooled_counts_equal_whole_collection
  /usr/local/lib/python3.13/dist-packages/rasterio/__init__.py:367: No

In [6]:
from pathlib import Path

from cloudband.baselines import ocm
from cloudband.colab.session import reload_package, start
from cloudband.datasets import cloudsen12 as ds
from cloudband.pipelines import phase0

reload_package("cloudband")

In [7]:
def report(position, total):
    if position % 25 == 0 or position == total:
        print(f"{position}/{total}", flush=True)

In [8]:
session = start(PROJECT_DIR, require_accelerator=True)

RuntimeError: no CUDA device; switch the Colab runtime to GPU

In [ ]:
table = phase0.load_test_split()
print(f"scenes: {len(table)}")
print("pairable:", ds.expected_pairable_scenes(table))

In [9]:
LIMIT = 5

ocm.check_version()
config = ocm.InferenceConfig(model_version=ocm.LATEST_MODEL_VERSION)

result = phase0.run(
    table,
    lambda stack: ocm.predict_array(stack, config),
    model_id=f"ocm-rgn-published-v{ocm.package_version()}",
    limit=LIMIT,
    progress=report,
)
result.scores

NameError: name 'table' is not defined

In [ ]:
PIXBOX_S2_V1_7_0 = {"clear": 92.42, "cloud": 91.52, "shadow": 81.37}

print("pairable scenes:", result.pairable)
phase0.compare_to_reference(result, PIXBOX_S2_V1_7_0)

In [ ]:
paths = phase0.save(
    result,
    Path("results/reports"),
    config=config.as_kwargs(),
    package_versions=session.package_versions,
)
for name, path in paths.items():
    print(f"{name}: {path}")